In [2]:
import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from ete3 import Tree
from collections import Counter
import subprocess

In [4]:
idf=pd.read_csv("SI tables/SI_table_7_v9.tsv",sep="\t",index_col=0)
td=list(idf.index)

In [13]:
import os
os.mkdir("outputs/all_topology_tests")

In [6]:
##Obtains subs model form previously run ML search in IQ-TREE
def get_subs_model(interv):
    f=open(f"outputs/phylogenetic_data_filtered/{interv}/ml_tree.iqtree","r").readlines()
    return [x for x in f if 'Model of substitution:' in x][0].split(":")[1].strip()

In [7]:
from Bio import SeqIO
##obtained trimmed MSA, standardize sequence names
def get_trimmed_msa(interv):
    
    fasta_file = f"outputs/phylogenetic_data_filtered/{interv}/trimmed_MSA.fasta"
    
    # Load into a dictionary: {sequence_id: sequence_string}
    fasta_dict = {record.id: str(record.seq) for record in SeqIO.parse(fasta_file, "fasta")}
    
    f2={}
    for x in fasta_dict:
        if 'GCA_' in x or 'GCF_' in x:
            f2[x.split(";")[1]]=fasta_dict[x]
        else:
            f2[x.split(";")[0]]=fasta_dict[x]
    
    from Bio.Seq import Seq
    from Bio.SeqRecord import SeqRecord
    # Convert dict back to SeqRecord objects
    records = [
        SeqRecord(Seq(seq), id=seq_id, description="") 
        for seq_id, seq in f2.items()
    ]
    
    # Write to a new FASTA file
    output_file = f"outputs/all_topology_tests/{interv}/trimmed_MSA_edit_names.fasta"
    SeqIO.write(records, output_file, "fasta")
    return

    

In [8]:
import subprocess

In [5]:
### takes a dataframe row and returns an origin annotation 
def tax_label(row):
    x=row['lineage']
    if 'Fungi' in x:
        return 'Fungi'
    elif 'Viruses' in x:
        return 'Viruses'
    elif 'Bacteria' in x:
        return 'Bacteria'
    elif 'Viridiplantae' in x:
        return 'Viridiplantae'
    elif 'primary_chimera' in str(row):
        return 'Arthropoda'
    elif 'Arthropoda' in x:
        return 'Arthropoda'
    elif 'Metazoa' in x:
        return 'Metazoa' 
    elif 'Sar' in x:
        return 'Sar'
    elif 'Evosea' in x:
        return 'Evosea'
    elif 'Metamonada' in x:
        return x
    elif 'Archaea' in x:
        return 'Archaea'
    else:
        return x
        

In [9]:
##write the Newick tree enforcing metazoan monophyly
def write_constraint(interv):
    t=Tree(f'outputs/phylogenetic_data_filtered/{interv}/final_rooted.tree')
    combined=pd.read_csv(f'outputs/phylogenetic_data_filtered/{interv}/combined_sequences_data.tsv',sep='\t')
    try:
        combined=combined.set_index('target_name')
    except:
        combined=combined.set_index('sseqid')
    
    tax_map={}
    for index, row in combined.iterrows():
        if ';' in index:
            tax_map[index.split(";")[1]]=tax_label(row)
        else:
            tax_map[index]=tax_label(row)
    
    for node in t:
        node.add_features(taxname=tax_map[node.name])

    clade=set()
    for node in t:
        if node.taxname in(['Arthropoda', 'Metazoa']):
            clade.add(node.name)

    metazoan=str(clade).replace("{","(").replace("}",")").replace("'","").replace(" ","")
    non_metazoan=set([x.name for x in t])-set(clade)
    non_metazoan=str(non_metazoan).replace("{","(").replace("}",")").replace("'","").replace(" ","")
    f=open(f"outputs/all_topology_tests/{interv}/constraint.txt","w")
    f.write(f"({metazoan},{non_metazoan});")
    f.close()
    return

In [21]:
## collect all trees with multi-sequence non-arthropod metazoan clades elsewhere in the tree
multi_meta=set([x.replace(" ","") for x in idf[idf["topological_category"].isin(["III","IV"])].index])

In [42]:
for interv in multi_meta:
    if interv not in os.listdir(f"outputs/all_topology_tests"):
        os.mkdir(f"outputs/all_topology_tests/{interv}")
        write_constraint(interv)
        get_trimmed_msa(interv)

In [ ]:
## run constrianed topology search with appropriate substitution model
for interv in multi_meta:
    if 'topo_tests.iqtree' not in os.listdir(f"outputs/all_topology_tests/"):
        subs_model=get_subs_model(interv)
        subprocess.run(['sbatch','constrained_topo_search_all.sh', interv,subs_model])


In [25]:
## for copy over unrooted treefile
for interv in multi_meta:
    ch=';'.join(interv.split(';')[0:2])
    t=f"outputs/phylogenetic_dataset/{ch}/{interv}/edited_ml_tree.treefile"
    if  interv in os.listdir("outputs/all_topology_tests") and 'topo_tests.trees' not in os.listdir(f"outputs/all_topology_tests/{interv}"):
        subprocess.run(['cp',t,f"outputs/all_topology_tests/{interv}/unconstrained_ml.treefile"])
##concatenate unconstrained and constrained trees together
for interv in multi_meta:
    if  interv in os.listdir("outputs/all_topology_tests") and 'topo_tests.trees' not in os.listdir(f"outputs/all_topology_tests/{interv}"):
        t1 = f"outputs/all_topology_tests/{interv}/unconstrained_ml.treefile"
        t2 = f"outputs/all_topology_tests/{interv}/{interv}.treefile"
        c  = f"outputs/all_topology_tests/{interv}/combo.treels"
     
        with open(c, "w") as outfile:
            with open(t1) as f1:
                outfile.write(f1.read())
            outfile.write("\n")
            with open(t2) as f2:
                outfile.write(f2.read())

In [24]:
import subprocess
## following completion of a topo search, run topology test
for interv in multi_meta:
    if  interv in os.listdir("outputs/all_topology_tests") and 'topo_tests.trees' not in os.listdir(f"outputs/all_topology_tests/{interv}"):
        subs=get_subs_model(interv)
        subprocess.run(['sbatch','constrained_topo_test_all.sh',interv,subs])

## Visualize constraint tree in iTOL

In [27]:
## copy over annotation files to visualize in iTOL
for interv in multi_meta:
    if 'topo_tests.trees' not in os.listdir(f"outputs/all_topology_tests/{interv}"):
        tc=os.listdir(f"outputs/phylogenetic_data_filtered/{interv}")
        for x in tc:
            if ".txt" in x:
                subprocess.run(["cp",f"outputs/phylogenetic_data_filtered/{interv}/{x}",f"outputs/all_topology_tests/{interv}/"])
        

In [11]:
import sys
import itolapi
from itolapi import Itol
from pathlib import Path
"""
Upload tree and associated annotation files to itol
"""
def upload_to_itol(interv):
  
    current_dir = Path(os.path.realpath(f"outputs/all_topology_tests/{interv}/{interv}.treefile")).parent
    tree = current_dir/f"{interv}.treefile"


    root_path = current_dir.parent
    sys.path.append(str(root_path))
    test = Itol()

    # Set the tree file


    test.add_file(current_dir / 'itol_color_strip.txt')
    test.add_file(current_dir / 'itol_taxonomic_info.txt')
    test.add_file(tree)
    # Add parameters

    test.params['APIkey'] = 'KEY'
    test.params['treeName'] = interv

    test.params['projectName'] ="Constrained_metazoan_monophyly"
    test.params['workspaceName'] = "Arthropod_chimeric_HGT_ML_trees"

    # Submit the tree, return false if fail
    return test.upload()

In [19]:
import os
for interv in multi_meta:
    if  interv in os.listdir("outputs/all_topology_tests") and 'topo_tests.trees' not in os.listdir(f"outputs/all_topology_tests/{interv}"):
        upload_to_itol(interv)

## Collect data from topo tests in a dataframe

In [7]:
from pathlib import Path
import re
import pandas as pd

##written by RK with chatGPT
def parse_topo_tests_iqtree(filepath):
    """
    Parse a topo_tests.iqtree file that contains exactly two trees.
    Returns a dict with columns suffixed _1 and _2.
    """
    text = Path(filepath).read_text(errors="replace")
    lines = text.splitlines()

    # Find header line
    header_idx = None
    for i, line in enumerate(lines):
        if line.strip().startswith("Tree") and "logL" in line:
            header_idx = i
            break

    if header_idx is None:
        raise ValueError(f"No topo test table found in {filepath}")

    # Data starts after dashed line
    data_lines = []
    for line in lines[header_idx + 1:]:
        if re.match(r"^\s*-{5,}", line):
            continue
        if re.match(r"^\s*\d+\s+", line):
            data_lines.append(line)
        elif data_lines:
            break

    if len(data_lines) != 2:
        raise ValueError(f"Expected 2 trees in {filepath}, found {len(data_lines)}")

    parsed = {}

    for line in data_lines:
        # Remove '+' markers and split
        tokens = [t for t in re.split(r"\s+", line.strip()) if t and t != "+" and t!='-']
        tree_id = tokens[0]  # "1" or "2"

        parsed[f"logL_{tree_id}"] = float(tokens[1])
        parsed[f"deltaL_{tree_id}"] = float(tokens[2])
        parsed[f"bp_RELL_{tree_id}"] = float(tokens[3])
        parsed[f"p_KH_{tree_id}"] = float(tokens[4])
        parsed[f"p_SH_{tree_id}"] = float(tokens[5])
        parsed[f"c_ELW_{tree_id}"] = float(tokens[6])
        parsed[f"p_AU_{tree_id}"] = float(tokens[7])

    return parsed


def collect_topo_tests(root_dir, filename="topo_tests.iqtree"):
    """
    Walk root_dir, parse each topo_tests.iqtree,
    return dataframe indexed by subdirectory name.
    """
    root = Path(root_dir)
    records = []
    index = []

    for fp in root.rglob(filename):
        subdir = fp.parent.name
        try:
            record = parse_topo_tests_iqtree(fp)
            records.append(record)
            index.append(subdir)
        except Exception as e:
            print(f"Skipping {fp}: {e}")

    df = pd.DataFrame(records, index=index)
    df.index.name = "subdir"
    return df.sort_index()


root_directory = "outputs/all_topology_tests"
df = collect_topo_tests(root_directory)


In [24]:
idf.index=[x.replace(" ","") for x in idf.index]

In [32]:
for index, row in df.iterrows():
    df.loc[index,'cluster_id']=int(idf.loc[index,'cluster_id'])

In [33]:
df['reject_monophyly']=df.p_AU_2<.05

In [34]:
df_filtered=df.loc[:,['logL_1','logL_2','deltaL_2','p_AU_1','p_AU_2','reject_monophyly','cluster_id']]

In [36]:
df_filtered.to_csv("outputs/all_tree_topology_tests.csv")

In [28]:
df_filtered=pd.read_csv("outputs/all_tree_topology_tests.csv",index_col=0)

In [29]:
df_filtered[df_filtered.reject_monophyly].shape[0],df_filtered.shape[0]

(38, 50)

In [30]:
df_filtered[df_filtered['reject_monophyly']].shape[0]

38

In [67]:
## collect for itol
for x in os.listdir("outputs/all_topology_tests"):
    !mkdir -p all_topology_tests/"$x"
    for y in os.listdir(f"outputs/all_topology_tests/{x}"):
        if '.iqtree' in y or '.treefile' in y:
            !cp outputs/all_topology_tests/"$x"/"$y" "all_topology_tests/$x"
        if y=='constraint.txt':
            !cp outputs/all_topology_tests/"$x"/"$y" "all_topology_tests/$x"

